## What is a Lexer?
A Lexer (short for Lexical Analyzer) is the first phase of a compiler. Its job is to read raw source code (as plain text) and convert it into a sequence of tokens that are meaningful for the next stage — the parser.

## What is a Parser?
The parser calls the lexer to get tokens. It processes tokens according to grammar rules (for, declarations, expressions) and then builds a nested AST that can be used for further analysis or code generation. It also reports syntax errors.

## What is Semantic Analysis?
Semantic analysis is the whole compiler phase after parsing that ensures the program is meaningful and consistent according to the language’s rules.
It checks things that parsing alone cannot enforce, such as:

- Variables must be declared before use
- Types must match in expressions
- Function calls must use the right number and type of arguments
- Return statements must match the function’s declared type
- Break/continue must only appear inside loops
- The program respects scope rules (inner vs outer variables)

Think of semantic analysis as:
“Does this syntactically valid program make sense in the language?”

### Type Checking
Type checking is a subset of semantic analysis. It focuses specifically on data types:

- Is an int being assigned to an int variable?
- Are you calling a function that returns a string where an int is expected?
- Are you adding an int to a float, and if so, is coercion allowed?
- Does the return type of a function match its declaration?

So type checking = semantic analysis about types only. In a real compiler, the type checker works like this:

- Walk the AST.
- Look up variable types in the symbol table.
- Infer/check expression types.
- Raise errors when things don’t match.


In [ ]:
Before we dive in.. Some good resources:
1. 2019 EuroLLVM Developers’ Meeting: V. Bridgers & F. Piovezan “LLVM IR Tutorial - Phis, GEPs ...”
2. Stanford Compilers Course- Alex Aiken - https://web.stanford.edu/class/archive/cs/cs143/cs143.1128/
3. https://www.youtube.com/watch?v=zclxRbh4AN0
4. LLVM Kaleidoscope: Implementing a language with LLVM

# Lexical analysis

In [3]:
import re

TOKEN_SPEC = [
    ('COMMENT_SINGLE', r'//.*'),          # Single-line comment
    ('ML_COMMENT_START', r'/\*'),         # Multi-line comment start
    ('STRING_START', r'"'),                # String literal start (double quotes)
    ('NUMBER',    r'\d+'),                
    ('ID',        r'[A-Za-z_]\w*'),      
    ('ASSIGN',    r'='),                  
    ('SEMI',      r';'),                  
    ('PUNC',      r'[,.]'),               
    ('LBRACS',    r'[\(\[\{]'),           
    ('RBRACS',    r'[\)\]\}]'),           
    ('OP',        r'[+\-*/<>=!]+'),       
    ('SKIP',      r'[ \t]+'),             
    ('NEWLINE',   r'\n'),                 
    ('MISMATCH',  r'.'),                  
]

token_re = re.compile('|'.join(f'(?P<{name}>{regex})' for name, regex in TOKEN_SPEC))

def lexer(code):
    line_num = 1
    line_start = 0
    pos = 0
    length = len(code)
    
    while pos < length:
        match = token_re.match(code, pos)
        if not match:
            raise SyntaxError(f'Unexpected character {code[pos]!r} at line {line_num}, col {pos - line_start + 1}')
        
        kind = match.lastgroup
        value = match.group()
        start = match.start()

        if kind == 'NEWLINE':
            line_num += 1
            line_start = match.end()
            pos = match.end()
            continue
        elif kind == 'SKIP':
            pos = match.end()
            continue
        elif kind == 'COMMENT_SINGLE':
            pos = match.end()
            yield kind, value, line_num, start - line_start + 1
            continue
        elif kind == 'ML_COMMENT_START':
            end_pos = code.find('*/', match.end())
            if end_pos == -1:
                col = start - line_start + 1
                raise SyntaxError(f'Unterminated multi-line comment at line {line_num}, col {col}')
            comment_text = code[pos:end_pos+2]
            newlines = comment_text.count('\n')
            if newlines > 0:
                line_num += newlines
                # Update line_start to position after last newline in comment
                last_newline_pos = comment_text.rfind('\n')
                line_start = pos + last_newline_pos + 1
            pos = end_pos + 2
            yield 'COMMENT_MULTI', comment_text, line_num, start - line_start + 1
            continue
        elif kind == 'STRING_START':
            # Find closing quote
            end_pos = pos + 1
            escaped = False
            while end_pos < length:
                c = code[end_pos]
                if c == '\\' and not escaped:
                    escaped = True
                elif c == '"' and not escaped:
                    break
                else:
                    escaped = False
                end_pos += 1
            else:
                col = start - line_start + 1
                raise SyntaxError(f'Unterminated string literal at line {line_num}, col {col}')
            string_text = code[pos:end_pos+1]
            # Count newlines inside string (rare but possible with \n escapes or multiline strings)
            newlines = string_text.count('\n')
            if newlines > 0:
                line_num += newlines
                last_newline_pos = string_text.rfind('\n')
                line_start = pos + last_newline_pos + 1
            pos = end_pos + 1
            yield 'STRING', string_text, line_num, start - line_start + 1
            continue
        elif kind == 'MISMATCH':
            col = start - line_start + 1
            raise SyntaxError(f'Unexpected character {value!r} at line {line_num}, col {col}')
        else:
            if kind == 'NUMBER':
                value = int(value)
            elif kind == 'ID' and value in ('int', 'if', 'else', 'for'):
                kind = 'KEYWORD'
            col = start - line_start + 1
            yield kind, value, line_num, col
        
        pos = match.end()

# Example test: unterminated string

code = '''
for (int i=0; i<100; i=i+1) { // This comment never ends
  a[i] = a[i+1] + 1;
}
'''

try:
    for token in lexer(code):
        print(token)
except SyntaxError as e:
    print("Lexer Error:", e)


('KEYWORD', 'for', 2, 1)
('LBRACS', '(', 2, 5)
('KEYWORD', 'int', 2, 6)
('ID', 'i', 2, 10)
('ASSIGN', '=', 2, 11)
('NUMBER', 0, 2, 12)
('SEMI', ';', 2, 13)
('ID', 'i', 2, 15)
('OP', '<', 2, 16)
('NUMBER', 100, 2, 17)
('SEMI', ';', 2, 20)
('ID', 'i', 2, 22)
('ASSIGN', '=', 2, 23)
('ID', 'i', 2, 24)
('OP', '+', 2, 25)
('NUMBER', 1, 2, 26)
('RBRACS', ')', 2, 27)
('LBRACS', '{', 2, 29)
('COMMENT_SINGLE', '// This comment never ends', 2, 31)
('ID', 'a', 3, 3)
('LBRACS', '[', 3, 4)
('ID', 'i', 3, 5)
('RBRACS', ']', 3, 6)
('ASSIGN', '=', 3, 8)
('ID', 'a', 3, 10)
('LBRACS', '[', 3, 11)
('ID', 'i', 3, 12)
('OP', '+', 3, 13)
('NUMBER', 1, 3, 14)
('RBRACS', ']', 3, 15)
('OP', '+', 3, 17)
('NUMBER', 1, 3, 19)
('SEMI', ';', 3, 20)
('RBRACS', '}', 4, 1)


# Parser

In [4]:
class Parser:
    def __init__(self, tokens):
        self.tokens = [t for t in tokens if t[0] != 'COMMENT_SINGLE' and t[0] != 'COMMENT_MULTI']
        self.pos = 0

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return ('EOF', None, None, None)

    def consume(self, expected_kind=None, expected_value=None):
        tok = self.peek()
        if expected_kind and tok[0] != expected_kind:
            raise SyntaxError(f"Expected {expected_kind} but got {tok[0]} at line {tok[2]}, col {tok[3]}")
        if expected_value and tok[1] != expected_value:
            raise SyntaxError(f"Expected {expected_value} but got {tok[1]} at line {tok[2]}, col {tok[3]}")
        self.pos += 1
        return tok

    def parse(self):
        stmts = []
        while self.peek()[0] != 'EOF':
            stmts.append(self.statement())
        return ('program', stmts)

    def statement(self):
        tok = self.peek()
        if tok[0] == 'KEYWORD':
            if tok[1] == 'for':
                return self.for_loop()
            elif tok[1] == 'if':
                return self.if_statement()
            elif tok[1] == 'while':
                return self.while_statement()
            elif tok[1] == 'return':
                return self.return_statement()
            elif tok[1] == 'int':
                return self.declaration()
        elif tok[0] == 'LBRACS' and tok[1] == '{':
            return self.block()
        else:
            return self.expr_statement()

    def for_loop(self):
        self.consume('KEYWORD', 'for')
        self.consume('LBRACS', '(')
        init = self.for_init()
        self.consume('SEMI')
        cond = self.expression()
        self.consume('SEMI')
        step = self.expression()
        self.consume('RBRACS', ')')
        body = self.statement()
        return ('for', init, cond, step, body)

    def for_init(self):
        tok = self.peek()
        if tok[0] == 'KEYWORD' and tok[1] == 'int':
            decls = []
            while True:
                decl = self.declaration_no_semi()
                decls.append(decl)
                if self.peek()[0] == 'PUNC' and self.peek()[1] == ',':
                    self.consume('PUNC', ',')
                else:
                    break
            return ('for_init_decls', decls)
        elif tok[0] == 'SEMI':
            return None
        else:
            expr = self.expression()
            return ('for_init_expr', expr)

    def declaration_no_semi(self):
        vartype = self.consume('KEYWORD')[1]
        varname = self.consume('ID')[1]
        expr = None
        if self.peek()[0] == 'ASSIGN':
            self.consume('ASSIGN')
            expr = self.expression()
        return ('declaration', vartype, varname, expr)

    def if_statement(self):
        self.consume('KEYWORD', 'if')
        self.consume('LBRACS', '(')
        cond = self.expression()
        self.consume('RBRACS', ')')
        then_branch = self.statement()
        else_branch = None
        if self.peek()[0] == 'KEYWORD' and self.peek()[1] == 'else':
            self.consume('KEYWORD', 'else')
            else_branch = self.statement()
        return ('if', cond, then_branch, else_branch)

    def while_statement(self):
        self.consume('KEYWORD', 'while')
        self.consume('LBRACS', '(')
        cond = self.expression()
        self.consume('RBRACS', ')')
        body = self.statement()
        return ('while', cond, body)

    def return_statement(self):
        self.consume('KEYWORD', 'return')
        expr = None
        if self.peek()[0] != 'SEMI':
            expr = self.expression()
        self.consume('SEMI')
        return ('return', expr)

    def declaration(self):
        vartype = self.consume('KEYWORD')[1]
        varname = self.consume('ID')[1]
        expr = None
        if self.peek()[0] == 'ASSIGN':
            self.consume('ASSIGN')
            expr = self.expression()
        self.consume('SEMI')
        return ('declaration', vartype, varname, expr)

    def block(self):
        self.consume('LBRACS', '{')
        stmts = []
        while not (self.peek()[0] == 'RBRACS' and self.peek()[1] == '}'):
            stmts.append(self.statement())
        self.consume('RBRACS', '}')
        return ('block', stmts)

    def expr_statement(self):
        expr = self.expression()
        self.consume('SEMI')
        return ('expr_stmt', expr)

    # Expressions with comparison operators

    def expression(self):
        return self.assignment()

    def assignment(self):
        node = self.comparison()
        if self.peek()[0] == 'ASSIGN':
            self.consume('ASSIGN')
            right = self.assignment()
            return ('assign', node, right)
        return node

    def comparison(self):
        node = self.add_sub()
        while True:
            tok = self.peek()
            if tok[0] == 'OP' and tok[1] in ('<', '>', '<=', '>=', '==', '!='):
                op = tok[1]
                self.consume('OP')
                right = self.add_sub()
                node = ('binop', op, node, right)
            else:
                break
        return node

    def add_sub(self):
        node = self.mul_div()
        while True:
            tok = self.peek()
            if tok[0] == 'OP' and tok[1] in ('+', '-'):
                op = tok[1]
                self.consume('OP')
                right = self.mul_div()
                node = ('binop', op, node, right)
            else:
                break
        return node

    def mul_div(self):
        node = self.unary()
        while True:
            tok = self.peek()
            if tok[0] == 'OP' and tok[1] in ('*', '/'):
                op = tok[1]
                self.consume('OP')
                right = self.unary()
                node = ('binop', op, node, right)
            else:
                break
        return node

    def unary(self):
        tok = self.peek()
        if tok[0] == 'OP' and tok[1] in ('+', '-', '++', '--'):
            op = tok[1]
            self.consume('OP')
            node = self.unary()
            return ('unop', op, node)
        else:
            return self.primary()

    def primary(self):
        tok = self.peek()
        if tok[0] == 'NUMBER':
            self.consume('NUMBER')
            return ('num', tok[1])
        elif tok[0] == 'ID':
            id_name = tok[1]
            self.consume('ID')
            # function call
            if self.peek()[0] == 'LBRACS' and self.peek()[1] == '(':
                self.consume('LBRACS', '(')
                args = []
                if not (self.peek()[0] == 'RBRACS' and self.peek()[1] == ')'):
                    while True:
                        args.append(self.expression())
                        if self.peek()[0] == 'PUNC' and self.peek()[1] == ',':
                            self.consume('PUNC', ',')
                        else:
                            break
                self.consume('RBRACS', ')')
                return ('call', id_name, args)
            # array access
            elif self.peek()[0] == 'LBRACS' and self.peek()[1] == '[':
                self.consume('LBRACS', '[')
                index_expr = self.expression()
                self.consume('RBRACS', ']')
                return ('array_access', id_name, index_expr)
            else:
                return ('id', id_name)
        elif tok[0] == 'LBRACS' and tok[1] == '(':
            self.consume('LBRACS', '(')
            node = self.expression()
            self.consume('RBRACS', ')')
            return node
        else:
            raise SyntaxError(f"Unexpected token {tok} at line {tok[2]}, col {tok[3]}")


In [5]:
tokens = list(lexer(code))
parser = Parser(tokens)
ast = parser.parse()

import pprint
pprint.pprint(ast)


('program',
 [('for',
   ('for_init_decls', [('declaration', 'int', 'i', ('num', 0))]),
   ('binop', '<', ('id', 'i'), ('num', 100)),
   ('assign', ('id', 'i'), ('binop', '+', ('id', 'i'), ('num', 1))),
   ('block',
    [('expr_stmt',
      ('assign',
       ('array_access', 'a', ('id', 'i')),
       ('binop',
        '+',
        ('array_access', 'a', ('binop', '+', ('id', 'i'), ('num', 1))),
        ('num', 1))))]))])


# Semantic analyzer
Builds simple AST node classes for functions, variables, assignments, returns, calls, binary ops, ints and bools.

Performs a two-pass check:

Collect top-level declarations (global vars and function signatures).

For each function, check the body using an explicit loop and evaluate expressions using a non-recursive post-order evaluator (stack with visited flag).

Detects: undeclared vars, redeclarations, wrong operand types, argument count/ types mismatches, incorrect return types.

In [6]:
from collections import deque, namedtuple

# -------------------------
# AST node definitions
# -------------------------
class Node: pass

class Program(Node):
    def __init__(self, decls):
        self.decls = decls  # list of Decl or Stmt

class VarDecl(Node):
    def __init__(self, name, typ):
        self.name = name
        self.typ = typ      # string like "int" or "bool"

class FuncDecl(Node):
    def __init__(self, name, params, ret_type, body):
        # params: list of (param_name, type_str)
        self.name = name
        self.params = params
        self.ret_type = ret_type  # string or None
        self.body = body          # list of Stmts

class Assign(Node):
    def __init__(self, name, expr):
        self.name = name
        self.expr = expr

class Return(Node):
    def __init__(self, expr):
        self.expr = expr

# Expressions
class Expr(Node):
    pass

class Num(Expr):
    def __init__(self, value):
        self.value = value

class BoolLit(Expr):
    def __init__(self, value):
        self.value = value

class Var(Expr):
    def __init__(self, name):
        self.name = name

class BinaryOp(Expr):
    def __init__(self, op, left, right):
        self.op = op    # e.g. '+', '-', '==', 'and'
        self.left = left
        self.right = right

class Call(Expr):
    def __init__(self, fn_name, args):
        self.fn_name = fn_name
        self.args = args  # list of Expr


# -------------------------
# Type system helpers
# -------------------------
PRIMITIVE_TYPES = {"int", "bool"}

def is_compatible(actual, expected):
    """Simple compatibility: exact equality for primitives and same named types."""
    return actual == expected

# -------------------------
# Semantic Analyzer (non-recursive traversal)
# -------------------------
class SemanticError(Exception):
    pass

class SymbolTable:
    """Simple stack of dicts for scopes."""
    def __init__(self):
        self.scopes = [{}]

    def push(self):
        self.scopes.append({})

    def pop(self):
        self.scopes.pop()

    def declare(self, name, typ):
        if name in self.scopes[-1]:
            raise SemanticError(f"Redeclaration of '{name}' in the same scope.")
        self.scopes[-1][name] = typ

    def lookup(self, name):
        for s in reversed(self.scopes):
            if name in s:
                return s[name]
        return None

class SemanticAnalyzerIter:
    def __init__(self):
        self.globals = SymbolTable()
        # functions: map name -> (param_types_list, return_type)
        self.functions = {}

    def analyze(self, program: Program):
        # 1. First pass over top-level declarations to collect globals and function signatures.
        # Non-recursive: simple queue
        q = deque(program.decls)
        while q:
            node = q.popleft()
            if isinstance(node, VarDecl):
                # declare global variable
                if node.typ not in PRIMITIVE_TYPES:
                    raise SemanticError(f"Unknown type '{node.typ}' for global '{node.name}'")
                self.globals.declare(node.name, node.typ)
            elif isinstance(node, FuncDecl):
                # collect function signature
                if node.name in self.functions:
                    raise SemanticError(f"Redeclaration of function '{node.name}'")
                param_types = []
                for pname, ptype in node.params:
                    if ptype not in PRIMITIVE_TYPES:
                        raise SemanticError(f"Unknown type '{ptype}' for parameter '{pname}' in function '{node.name}'")
                    param_types.append(ptype)
                # allow None return type means "void"
                if node.ret_type is not None and node.ret_type not in PRIMITIVE_TYPES:
                    raise SemanticError(f"Unknown return type '{node.ret_type}' for function '{node.name}'")
                self.functions[node.name] = (param_types, node.ret_type)
            else:
                raise SemanticError(f"Top-level unsupported node: {type(node)}")

        # 2. Second pass: type-check each function body (and top-level statements if you had any)
        for decl in program.decls:
            if isinstance(decl, FuncDecl):
                self._check_function(decl)

    # ---------- expression type evaluator (non-recursive, post-order) ----------
    def eval_expr_type(self, expr, local_symtab: SymbolTable):
        """
        Evaluate expression type using an explicit stack (post-order).
        Returns a type string like 'int' or 'bool', or raises SemanticError.
        """
        # we will annotate nodes with a temporary 'checked_type' attribute
        stack = [(expr, False)]
        while stack:
            node, visited = stack.pop()
            if not visited:
                stack.append((node, True))
                # push children
                if isinstance(node, Num) or isinstance(node, BoolLit) or isinstance(node, Var):
                    pass
                elif isinstance(node, BinaryOp):
                    stack.append((node.right, False))
                    stack.append((node.left, False))
                elif isinstance(node, Call):
                    for a in reversed(node.args):
                        stack.append((a, False))
                else:
                    raise SemanticError(f"Unknown expr node in evaluator: {type(node)}")
            else:
                # process node: children already annotated
                if isinstance(node, Num):
                    node.checked_type = "int"
                elif isinstance(node, BoolLit):
                    node.checked_type = "bool"
                elif isinstance(node, Var):
                    t = local_symtab.lookup(node.name)
                    if t is None:
                        t = self.globals.lookup(node.name)
                    if t is None:
                        raise SemanticError(f"Undeclared variable '{node.name}'")
                    node.checked_type = t
                elif isinstance(node, BinaryOp):
                    lt = node.left.checked_type
                    rt = node.right.checked_type
                    # simple rules: +,-,*,/ take ints, result int
                    if node.op in {"+", "-", "*", "/"}:
                        if lt != "int" or rt != "int":
                            raise SemanticError(f"Operator '{node.op}' requires int operands (got {lt}, {rt})")
                        node.checked_type = "int"
                    elif node.op in {"==", "!="}:
                        # allow comparison of same primitive types
                        if lt != rt:
                            raise SemanticError(f"Operator '{node.op}' requires same-type operands (got {lt}, {rt})")
                        node.checked_type = "bool"
                    elif node.op in {"and", "or"}:
                        if lt != "bool" or rt != "bool":
                            raise SemanticError(f"Logical op '{node.op}' requires bool operands (got {lt}, {rt})")
                        node.checked_type = "bool"
                    else:
                        raise SemanticError(f"Unknown binary operator '{node.op}'")
                elif isinstance(node, Call):
                    sig = self.functions.get(node.fn_name)
                    if sig is None:
                        raise SemanticError(f"Call to undeclared function '{node.fn_name}'")
                    param_types, ret_type = sig
                    if len(param_types) != len(node.args):
                        raise SemanticError(f"Function '{node.fn_name}' expects {len(param_types)} args, got {len(node.args)}")
                    # compare param types
                    for i, arg in enumerate(node.args):
                        atype = arg.checked_type
                        exp = param_types[i]
                        if not is_compatible(atype, exp):
                            raise SemanticError(f"Argument {i} of '{node.fn_name}' expects {exp}, got {atype}")
                    node.checked_type = ret_type if ret_type is not None else "void"
                else:
                    raise SemanticError("unreachable")
        return expr.checked_type

    # ---------- function body checker (non-recursive traversal of statements) ----------
    def _check_function(self, func: FuncDecl):
        # create function-local symbol table and pre-declare parameters
        local = SymbolTable()
        # copy global scopes into local so lookup falls back to globals via analyzer order
        # We'll use local_symtab.lookup then globals.lookup in eval_expr_type (already does that).
        # Pre-declare params in the local scope
        for pname, ptype in func.params:
            local.declare(pname, ptype)

        # traverse function body using explicit stack: statements in order
        # we treat function body as a simple list; for each statement, we might need expr-type evaluation
        for stmt in func.body:
            if isinstance(stmt, VarDecl):
                local.declare(stmt.name, stmt.typ)
            elif isinstance(stmt, Assign):
                # check var exists (in local or global)
                vtype = local.lookup(stmt.name)
                if vtype is None:
                    vtype = self.globals.lookup(stmt.name)
                if vtype is None:
                    raise SemanticError(f"Assignment to undeclared variable '{stmt.name}'")
                expr_type = self.eval_expr_type(stmt.expr, local)
                if not is_compatible(expr_type, vtype):
                    raise SemanticError(f"Type error in assignment to '{stmt.name}': expected {vtype}, got {expr_type}")
            elif isinstance(stmt, Return):
                if func.ret_type is None:
                    raise SemanticError(f"Return in void function '{func.name}'")
                expr_type = self.eval_expr_type(stmt.expr, local)
                if not is_compatible(expr_type, func.ret_type):
                    raise SemanticError(f"Return type mismatch in '{func.name}': expected {func.ret_type}, got {expr_type}")
            else:
                raise SemanticError(f"Unsupported statement kind {type(stmt)} in function '{func.name}'")

# -------------------------
# Example usage and tests
# -------------------------
if __name__ == "__main__":
    # Correct program
    prog = Program([
        VarDecl("g", "int"),
        FuncDecl(
            name="add_one",
            params=[("x", "int")],
            ret_type="int",
            body=[
                # int y; y = x + 1; return y;
                VarDecl("y", "int"),
                Assign("y", BinaryOp("+", Var("x"), Num(1))),
                Return(Var("y"))
            ]
        )
    ])

    analyzer = SemanticAnalyzerIter()
    analyzer.analyze(prog)
    print("Program 1: OK")

    # Program with errors
    prog2 = Program([
        FuncDecl(
            name="bad",
            params=[("b", "bool")],
            ret_type="int",
            body=[
                # tries to return a bool in a function declared returning int
                Return(Var("b"))
            ]
        )
    ])

    try:
        SemanticAnalyzerIter().analyze(prog2)
    except SemanticError as e:
        print("Program 2 error:", e)

    # Program with undeclared var usage
    prog3 = Program([
        FuncDecl(
            name="call_undeclared",
            params=[],
            ret_type=None,
            body=[
                Assign("x", Num(10))   # x not declared
            ]
        )
    ])
    try:
        SemanticAnalyzerIter().analyze(prog3)
    except SemanticError as e:
        print("Program 3 error:", e)


Program 1: OK
Program 2 error: Return type mismatch in 'bad': expected int, got bool
Program 3 error: Assignment to undeclared variable 'x'
